In [ ]:
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, Polygon, GeoData, GeoJSON, ImageOverlay
from fetchez.registry import ModuleRegistry, BundleRegistry
import os
import threading
import collections
import json
import geopandas as gpd
import logging
import fetchez.recipe
import fetchez.core
import base64
from io import BytesIO

# --- Header Block ---
header = widgets.HTML(
    "<h2 style='font-family: sans-serif; color: #333;'>🌐 Globato DEM Builder</h2><hr style='border: 1px solid #ddd;'/>"
)

# --- Load Registries & Extract Descriptions ---
ModuleRegistry.load_all()
BundleRegistry.load_all()
registry = ModuleRegistry.get_registry()
registry.update(BundleRegistry.get_registry())

globato_sources = {}
for name, meta in sorted(registry.items()):
    if "glob-stream" in meta.get("tags", []) and name not in meta.get("aliases", []):
        desc = meta.get("description") or meta.get("desc", "No description provided.")
        globato_sources[name] = desc.strip().split("\n")[0]

# --- Left Column: Map Selector ---
m = Map(
    center=[44.6, -124.05],
    zoom=10,
    layout=widgets.Layout(height="auto", flex="1", margin="0px 15px 0px 0px"),
)
draw_control = DrawControl(
    rectangle={"shapeOptions": {"color": "#0074D9", "weight": 2, "fillOpacity": 0.2}}
)
m.add(draw_control)

session_layers = []

# --- Right Column: Form Options (Styled for Prominence) ---
style = {"description_width": "100px"}
layout = widgets.Layout(width="100%", margin="0px 0px 10px 0px")

region_input = widgets.Text(
    description="Region:",
    placeholder="Auto-populated by map",
    style=style,
    layout=layout,
)
upload_widget = widgets.FileUpload(
    accept=".geojson,.gpkg",
    multiple=False,
    description="Upload Vector Region",
    layout=layout,
    style=style,
)
increment_input = widgets.Dropdown(
    options=["1s", "1/3s", "1/9s", "3s", "30m"],
    value="1s",
    description="Increment:",
    style=style,
    layout=layout,
)
srs_input = widgets.Dropdown(
    options=["EPSG:4326+3855", "EPSG:4269+5703", "EPSG:3857"],
    value="EPSG:4326+3855",
    description="Target SRS:",
    style=style,
    layout=layout,
)
buffer_input = widgets.IntSlider(
    value=5,
    min=0,
    max=100,
    step=1,
    description="Buffer %:",
    tooltip="Processing Buffer Percentage",
    style=style,
    layout=layout,
)

outname_input = widgets.Text(
    description="Output Name:", value="globato_dem", style=style, layout=layout
)
outdir_input = widgets.Text(
    description="Out Folder:", value="~/workshop/output", style=style, layout=layout
)
cache_input = widgets.Text(
    description="Cache Dir:",
    value="~/workshop/shared_cache",
    style=style,
    layout=layout,
)

source_checkboxes = []
for src_name, src_desc in globato_sources.items():
    cb = widgets.Checkbox(
        value=False, description=f"{src_name}", indent=False, tooltip=src_desc
    )
    source_checkboxes.append(cb)

sources_ui = widgets.VBox(
    [widgets.HTML("<h4 style='margin: 0px 0px 10px 0px;'>Data Sources</h4>")]
    + source_checkboxes,
    layout=widgets.Layout(
        max_height="200px",
        overflow="auto",
        border="1px solid #ccc",
        padding="10px",
        background_color="#ffffff",
        margin="0px 0px 15px 0px",
    ),
)

# --- Logging & UI Updates ---
output_log = widgets.Output()
log_display = widgets.HTML(
    value="<pre style='background:#1e1e1e; color:#50fa7b; padding:15px; height:200px; overflow-y:auto; border-radius: 5px; font-size: 13px;'></pre>"
)

fetchez.recipe.setup_logging = lambda *args, **kwargs: None
log_buffer = collections.deque(maxlen=150)


def ansi_to_html(text):
    text = text.replace("\033[0m", "</span>").replace("\x1b[0m", "</span>")
    colors = {
        "30": "#a8a8a8",
        "31": "#ff5555",
        "32": "#50fa7b",
        "33": "#f1fa8c",
        "34": "#bd93f9",
        "35": "#ff79c6",
        "36": "#8be9fd",
        "37": "#f8f8f2",
    }
    for code, color in colors.items():
        text = text.replace(f"\033[{code}m", f'<span style="color: {color};">')
        text = text.replace(f"\x1b[{code}m", f'<span style="color: {color};">')
    text = text.replace("\033[1m", '<span style="font-weight: bold; color: #ffffff;">')
    return text


class BoundedWidgetHandler(logging.Handler):
    def __init__(self, html_widget):
        super().__init__()
        self.widget = html_widget
        self.setFormatter(
            logging.Formatter("[ %(levelname)s ] %(module)s: %(message)s")
        )

    def emit(self, record):
        raw_msg = self.format(record)
        log_buffer.append(ansi_to_html(raw_msg))
        self.widget.value = f"<pre style='background:#1e1e1e; color:#d4d4d4; padding:15px; height:200px; overflow-y:auto; border-radius: 5px;'>{'<br>'.join(log_buffer)}</pre>"


root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
if not any(isinstance(h, BoundedWidgetHandler) for h in root_logger.handlers):
    root_logger.addHandler(BoundedWidgetHandler(log_display))

# --- Execution Buttons ---
btn_layout = widgets.Layout(height="40px", margin="5px 0px", width="100%")
build_button = widgets.Button(
    description=" Build DEM", button_style="success", icon="cogs", layout=btn_layout
)
preview_osm_button = widgets.Button(
    description=" Preview Topology",
    button_style="primary",
    icon="map",
    layout=btn_layout,
)
cancel_button = widgets.Button(
    description=" Cancel Build",
    button_style="danger",
    icon="stop",
    layout=btn_layout,
    disabled=True,
)
clear_button = widgets.Button(
    description=" Clear Map", button_style="info", icon="refresh", layout=btn_layout
)


# --- Upload Handler ---
def on_upload_change(change):
    if upload_widget.value:
        uploaded_file = upload_widget.value[0]
        file_name = uploaded_file["name"]
        with open(file_name, "wb") as f:
            f.write(uploaded_file["content"])
        region_input.value = file_name
        try:
            gdf = gpd.read_file(file_name)
            geo_layer = GeoData(
                geo_dataframe=gdf,
                style={"color": "black", "fillOpacity": 0.1, "weight": 2},
                name="Vector Upload",
            )
            m.add(geo_layer)
            session_layers.append(geo_layer)
            bounds = gdf.total_bounds
            m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
        except Exception as e:
            output_log.append_stdout(f"⚠️ Could not preview vector: {e}\n")


upload_widget.observe(on_upload_change, names="value")


# --- Render Hillshade on Map ---
def render_hillshade_overlay(hs_path, bounds):
    """Reads the side-car _hs.tif, converts to base64, and overlays on the map."""
    try:
        import rasterio
        import numpy as np
        from PIL import Image

        with rasterio.open(hs_path) as src:
            data = src.read()
            # Move bands to last dimension for PIL (H, W, C)
            img_data = np.moveaxis(data, 0, -1)
            img = Image.fromarray(img_data)

            buffered = BytesIO()
            img.save(buffered, format="PNG")
            img_str = base64.b64encode(buffered.getvalue()).decode()

            # bounds = (w, e, s, n) -> ImageOverlay requires [[s, w], [n, e]]
            w, e, s, n = bounds
            image_url = f"data:image/png;base64,{img_str}"
            overlay = ImageOverlay(
                url=image_url, bounds=[[s, w], [n, e]], name="DEM Hillshade"
            )

            m.add(overlay)
            session_layers.append(overlay)
    except Exception as e:
        output_log.append_stdout(f"⚠️ Failed to render hillshade on map: {e}\n")


# --- Topology Preview Handler ---
def topology_style(feature):
    colors = {
        "ocean": "#000080",
        "estuary": "#008080",
        "river": "#00FFFF",
        "lake": "#4169E1",
        "land": "#D2B48C",
        "breakwater": "#808080",
    }
    geom_class = feature["properties"].get("class", "land")
    return {
        "color": "black",
        "weight": 1,
        "fillColor": colors.get(geom_class, "#D2B48C"),
        "fillOpacity": 0.5,
    }


def run_topology_thread(region_str, outdir):
    try:
        import fetchez.api

        output_log.append_stdout(
            f"🌍 Fetching Topological OSM Mask for {region_str}...\n"
        )
        files = fetchez.api.get(
            "osm_landmask",
            region=region_str,
            outdir=os.path.expanduser(outdir),
            output_mode="topology",
        )
        if files:
            with open(files[0], "r") as f:
                geo_data = json.load(f)
            geo_layer = GeoJSON(
                data=geo_data, style_callback=topology_style, name="Topological Preview"
            )
            m.add(geo_layer)
            session_layers.append(geo_layer)
    except Exception as e:
        output_log.append_stdout(f"❌ Topology preview failed: {e}\n")
    finally:
        preview_osm_button.disabled = False


def on_preview_osm_clicked(b):
    if not region_input.value or "/" not in region_input.value:
        return
    preview_osm_button.disabled = True
    threading.Thread(
        target=run_topology_thread, args=(region_input.value, outdir_input.value)
    ).start()


preview_osm_button.on_click(on_preview_osm_clicked)


# --- Build Threading ---
def run_build_thread(sources, region, increment, buffer, srs, outname, outdir, cache):
    outdir = os.path.expanduser(outdir)
    cache = os.path.expanduser(cache)
    fetchez.core.STOP_EVENT.clear()
    completed_batches = []

    output_log.append_stdout(f"🚀 Starting background build for {region}...\n")
    try:
        import globato.api

        pipeline = globato.api.build(
            sources,
            region,
            increment,
            extend=f"0:{buffer}",
            t_srs=srs,
            outname=outname,
            outdir=outdir,
            shared_cache=cache,
        )
        for tile_data in pipeline:
            config, target_region, batch_name, abs_cache, base_out, tile_dir = tile_data
            completed_batches.append(batch_name)

            if target_region:
                w, e, s, n = target_region.to_list()
                true_filename = (
                    f"{outname}_{batch_name}_hs.tif"
                    if batch_name
                    else f"{outname}_hs.tif"
                )
                hs_path = os.path.join(tile_dir, true_filename)
                output_log.append_stdout(hs_path)
                output_log.append_stdout(os.path.exists(hs_path))

                # Render hillshade if it exists, otherwise fallback to bounding box
                if os.path.exists(hs_path):
                    render_hillshade_overlay(hs_path, (w, e, s, n))
                else:
                    completed_poly = Polygon(
                        locations=[(s, w), (n, w), (n, e), (s, e)],
                        color="blue",
                        fill_color="blue",
                        fill_opacity=0.2,
                    )
                    m.add(completed_poly)
                    session_layers.append(completed_poly)

            output_log.clear_output(wait=True)
            output_log.append_stdout(
                f"✅ Completed ({len(completed_batches)} tiles): {', '.join(completed_batches[-3:])}...\n"
            )

        output_log.append_stdout(
            "🎉 Entire DEM build process completed successfully!\n"
        )
    except Exception as e:
        if fetchez.core.STOP_EVENT.is_set():
            output_log.append_stdout("🛑 Pipeline cancelled.\n")
        else:
            output_log.append_stdout(f"❌ Pipeline failed: {e}\n")
    finally:
        build_button.disabled = False
        cancel_button.disabled = True


# --- General UI Handlers ---
def on_draw(target, action, geo_json):
    if action == "created":
        coords = geo_json["geometry"]["coordinates"][0]
        region_input.value = f"{min([p[0] for p in coords]):.5f}/{max([p[0] for p in coords]):.5f}/{min([p[1] for p in coords]):.5f}/{max([p[1] for p in coords]):.5f}"


def on_build_clicked(b):
    output_log.clear_output()
    selected_sources = [
        cb.description.replace("<b>", "").replace("</b>", "")
        for cb in source_checkboxes
        if cb.value
    ]
    if not selected_sources:
        return
    build_button.disabled = True
    cancel_button.disabled = False
    threading.Thread(
        target=run_build_thread,
        args=(
            selected_sources,
            region_input.value,
            increment_input.value,
            buffer_input.value,
            srs_input.value,
            outname_input.value,
            outdir_input.value,
            cache_input.value,
        ),
    ).start()


def on_clear_clicked(b):
    output_log.clear_output()
    log_buffer.clear()
    log_display.value = "<pre style='background:#1e1e1e; color:#d4d4d4; padding:15px; height:200px; overflow-y:auto; border-radius: 5px;'></pre>"
    for layer in session_layers:
        if layer in m.layers:
            m.remove(layer)
    session_layers.clear()
    draw_control.clear()


def on_cancel_clicked(b):
    fetchez.core.STOP_EVENT.set()
    cancel_button.disabled = True


draw_control.on_draw(on_draw)
build_button.on_click(on_build_clicked)
cancel_button.on_click(on_cancel_clicked)
clear_button.on_click(on_clear_clicked)

# Assemble Layout
form_container = widgets.VBox(
    [
        widgets.HTML(
            "<h3 style='margin-top:0px; border-bottom:1px solid #ddd;'>Spatial Configuration</h3>"
        ),
        upload_widget,
        region_input,
        increment_input,
        srs_input,
        buffer_input,
        widgets.HTML(
            "<h3 style='margin-top:15px; border-bottom:1px solid #ddd;'>Outputs</h3>"
        ),
        outname_input,
        outdir_input,
        cache_input,
        sources_ui,
        preview_osm_button,
        build_button,
        cancel_button,
        clear_button,
    ],
    layout=widgets.Layout(
        width="400px",
        padding="20px",
        background_color="#f8f9fa",
        border="1px solid #e0e0e0",
        border_radius="8px",
        box_shadow="0px 4px 6px rgba(0,0,0,0.1)",
    ),
)

dashboard = widgets.VBox(
    [
        header,
        widgets.HBox(
            [m, form_container],
            layout=widgets.Layout(width="100%", align_items="stretch"),
        ),
        widgets.HTML("<h3 style='margin-top:20px;'>Execution Logs</h3>"),
        output_log,
        log_display,
    ]
)

display(dashboard)